# CaseNote AI: run the evaluation

This notebook runs CaseNote AI on the 12 synthetic cases and scores it against the expert labels.

**Before you start:** add your API key as a Colab secret. Click the 🔑 key icon in the left sidebar, add a secret named `GEMINI_API_KEY` (or `ANTHROPIC_API_KEY`), paste the key as its value and turn on *Notebook access*. Never paste a key into a code cell.

All data in this project is synthetic.

In [7]:
# 1. Get the code
!git clone -q https://github.com/tugbasahingunes/casenote-ai.git
%cd casenote-ai
!pip install -q -r requirements.txt

/content/casenote-ai/casenote-ai


In [8]:
# 2. Choose the model provider and load the key from Colab secrets
import os
from google.colab import userdata

PROVIDER = "gemini"   # or "anthropic"
KEY_NAME = {"gemini": "GEMINI_API_KEY", "anthropic": "ANTHROPIC_API_KEY"}[PROVIDER]

os.environ["CASENOTE_PROVIDER"] = PROVIDER
os.environ[KEY_NAME] = userdata.get(KEY_NAME)
# Optional: pick a specific model, e.g. os.environ["CASENOTE_MODEL"] = "gemini-3.8-flash"
print("Provider:", PROVIDER)

Provider: gemini


In [9]:
# 3. Try one note first
!python src/casenote.py "Mrs. H (83) lives alone. Her nephew has her bank codes. When he stepped out she said quietly: 'Please don't tell him I said anything.'"

{
  "person": [
    {
      "domain": 4,
      "level": "record",
      "quote": "lives alone",
      "source": "professional_observed"
    },
    {
      "domain": 12,
      "level": "urgent_human_review",
      "quote": "Her nephew has her bank codes.",
      "source": "professional_observed"
    },
    {
      "domain": 12,
      "level": "urgent_human_review",
      "quote": "Please don't tell him I said anything.",
      "source": "person_said"
    }
  ],
  "caregiver": [],
  "young_carer": [],
  "no_info": [
    1,
    2,
    3,
    5,
    6,
    7,
    8,
    9,
    10,
    11,
    13
  ],
  "said_vs_observed": [],
  "removed_unverified_quotes": [],
  "model": "gemini-3.8-flash"
}


In [11]:
# 4. Run all 12 cases (about 2 minutes)
!python src/run_cases.py

Analysing C01...
Analysing C02...
Traceback (most recent call last):
  File "/content/casenote-ai/casenote-ai/src/run_cases.py", line 34, in <module>
    main()
    ~~~~^^
  File "/content/casenote-ai/casenote-ai/src/run_cases.py", line 25, in main
    preds[case["id"]] = analyze_note(case["note"], criteria)
                        ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/casenote-ai/casenote-ai/src/casenote.py", line 198, in analyze_note
    raw = call(note, criteria, model, client)
  File "/content/casenote-ai/casenote-ai/src/casenote.py", line 174, in _call_gemini
    response = client.models.generate_content(
        model=model,
    ...<5 lines>...
        ),
    )
  File "/usr/local/lib/python3.13/dist-packages/google/genai/models.py", line 6558, in generate_content
    response = self._generate_content(
        model=model, contents=contents, config=parsed_config_to_call
    )
  File "/usr/local/lib/python3.13/dist-packages/google/genai/models.py", line 4986, in _ge

In [5]:
# 5. Score against the expert labels
import glob
import os
latest = max(glob.glob("eval/results/predictions_*.json"), key=os.path.getmtime)
!python eval/evaluate.py {latest}

ValueError: max() iterable argument is empty

In [6]:
# 6. Download the predictions and report to upload to GitHub
from google.colab import files
name = latest.split("predictions_")[1].replace(".json", "")
files.download(latest)
files.download(f"eval/results/report_{name}.md")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>